In [ ]:
import os
import torch
import random
import numpy as np


In [ ]:
# Set a seed for reproducibility

def fix_seed(seed):
    # Fix for Python hash seed (to ensure reproducibility in Python operations)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # Fix for random seed (used by random module)
    random.seed(seed)
    
    # Fix for numpy random operations
    np.random.seed(seed)
    
    # Fix for PyTorch random seed (CPU and GPU)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    
    # Ensure deterministic behavior for CuDNN
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    # Additional configuration for CUBLAS for reproducibility
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    
    # For scanpy or any random process in other libraries
    # We can use a random_state argument where applicable, like in scanpy PCA, DEG, etc.

# Set the seed
seed = 42
fix_seed(seed)

In [ ]:
import re
import numpy as np

# --- regex helpers ---
_REP_SUFFIX = re.compile(r"(_\d+\+\d+)$")  # matches "_1+1" at end

def _strip_prefix_suffix_uns_key(full_key: str) -> str:
    """
    Convert adata.uns keys like:
      'K562(?)_ARHGAP22+ctrl_1+1' -> 'ARHGAP22+ctrl'
      'A549_AHR+FEV_1+1'         -> 'AHR+FEV'
      'A549_ctrl+BAK1_1+1'       -> 'ctrl+BAK1'
    """
    s = str(full_key)
    s = _REP_SUFFIX.sub("", s)          # drop trailing "_1+1"
    if "_" in s:
        s = s.split("_", 1)[1]          # drop cell-line prefix (everything before first "_")
    return s

def canonicalize_pert(pert: str, ctrl_token: str = "ctrl") -> str:
    """
    Canonicalize perturbation keys from either loader (e.g. 'AHR+ctrl', 'ctrl+BAK1')
    or from uns keys (after stripping prefix/suffix) to a stable representation.

    Rules:
      - Split by '+'
      - If ctrl is present, force ctrl to the end: 'ctrl+BAK1' -> 'BAK1+ctrl'
      - For 2-gene (or multi-gene) perturbations, sort remaining genes alphabetically
        to make order-invariant: 'FEV+AHR' -> 'AHR+FEV'
    """
    s = str(pert)

    parts = s.split("+")
    parts = [p for p in parts if p != ""]  # defensive

    if len(parts) <= 1:
        return s

    # normalize ctrl placement and ordering for other genes
    has_ctrl = ctrl_token in parts
    genes = [p for p in parts if p != ctrl_token]

    # sort non-ctrl genes to make order invariant
    genes_sorted = sorted(genes)

    if has_ctrl:
        return "+".join(genes_sorted + [ctrl_token])
    else:
        return "+".join(genes_sorted)

def build_canonical_de_dict(
    adata,
    de_key: str,
    out_key: str = None,
    ctrl_token: str = "ctrl",
) -> str:
    """
    Build and store a canonical DE dict:
      adata.uns[out_key][canonical_pert] = list_of_de_genes

    Returns: out_key used
    """
    if out_key is None:
        out_key = de_key + "_canon"

    if de_key not in adata.uns:
        raise KeyError(f"de_key='{de_key}' not found in adata.uns")

    de_full = adata.uns[de_key]  # dict: full_key -> de gene list
    canon_dict = {}
    collisions = {}

    for full_k, genes in de_full.items():
        core = _strip_prefix_suffix_uns_key(full_k)
        ck = canonicalize_pert(core, ctrl_token=ctrl_token)

        # if multiple full keys map to same canonical key, keep first and record collisions
        if ck in canon_dict:
            collisions.setdefault(ck, []).append(full_k)
            continue
        canon_dict[ck] = genes

    adata.uns[out_key] = canon_dict

    print(f"[DE canon] stored adata.uns['{out_key}'] with {len(canon_dict)} entries.")
    if collisions:
        print(f"[DE canon] {len(collisions)} canonical keys had multiple source full-keys (kept first).")
        # show a few collisions
        for ck, fulls in list(collisions.items())[:5]:
            print("  collision:", ck, "extra full-keys:", fulls[:5])

    return out_key

def remap_pert_cat_to_canonical(pert_cat, ctrl_token: str = "ctrl"):
    """
    Takes pert_cat array-like (strings) and canonicalizes each entry.
    """
    return np.array([canonicalize_pert(p, ctrl_token=ctrl_token) for p in pert_cat], dtype=object)

def report_mapping_coverage(pert_cat_canon, de_canon_keys):
    """
    Quick coverage report: how many pert_cat_canon are present in DE canonical dict.
    """
    pert_set = set(map(str, np.unique(pert_cat_canon)))
    de_set = set(map(str, de_canon_keys))
    missing = sorted(list(pert_set - de_set))
    print(f"[Coverage] unique perts in eval: {len(pert_set)} | present in DE dict: {len(pert_set) - len(missing)} | missing: {len(missing)}")
    if missing:
        print("  first10 missing:", missing[:10])
    return missing


In [ ]:
from gears import PertData
from CellPLM.pipeline.perturbation_prediction import PerturbationPredictionPipeline

data_dir = '/media/rokny/DATA2/Sally/data/perturb-seq'
data_name = 'replogle_k562_essential' #adamson, norman or replogle_k562_essential

split = 'simulation'
batch_size = 64

PRETRAIN_VERSION = '20231027_85M'
DEVICE = 'cuda:0'

pert_data = PertData(data_dir)
pert_data.load(data_name=data_name) #adamson, norman or replogle

pipe = PerturbationPredictionPipeline(
    pretrain_prefix=PRETRAIN_VERSION,
    pretrain_directory="../ckpt",
)

pert_data.prepare_split(split=split, seed=seed)

pert_data.get_dataloader(batch_size=batch_size, test_batch_size=batch_size)

train_loader_gears = pert_data.dataloader["train_loader"]
val_loader_gears = pert_data.dataloader["val_loader"]
test_loader_gears = pert_data.dataloader["test_loader"]

gears_train_dataset = train_loader_gears.dataset
gears_val_dataset   = val_loader_gears.dataset
gears_test_dataset  = test_loader_gears.dataset

batch_keymap = {
    "x": "x",
    "y": "y",
    "condition": "pert",   # GEARS uses 'pert' as the perturbation category label
    "pert_idx": "pert_idx" # use indices to set -100 directly
}

In [ ]:
pipe.fit(pert_data, 
         train_loader=train_loader_gears,
         val_loader=val_loader_gears,
         device=DEVICE, 
         batch_keymap=batch_keymap)

In [ ]:
res = pipe.evaluate_gears(
    pert_data=pert_data,
    test_loader=test_loader_gears,
    device=DEVICE,
)

In [ ]:
print(res)

In [ ]:
print(res["test_metrics"])

In [ ]:
def canon_pert(p, ctrl_token="ctrl"):
    """
    Canonicalise perturbation labels so different formats match:
    - remove trailing replicate suffix like _1+1
    - remove leading prefixes like A549_ / K562(?)_ when present
    - normalise ctrl pairing: ctrl+GENE -> GENE+ctrl
    - make multi-gene perturbations order-invariant (sorted)
    """
    s = str(p)

    # drop trailing replicate suffix "_1+1", "_2+1", etc
    s = re.sub(r"_\d+\+\d+$", "", s)

    # if it's like "A549_GENE+GENE" keep the last chunk after underscore
    if "_" in s:
        tail = s.split("_")[-1]
        if "+" in tail:
            s = tail

    if s == ctrl_token or "+" not in s:
        return s

    parts = [x.strip() for x in s.split("+") if x.strip()]

    # move ctrl to end for 1-gene+ctrl cases
    if ctrl_token in parts and len(parts) == 2:
        other = parts[0] if parts[1] == ctrl_token else parts[1]
        return f"{other}+{ctrl_token}"

    # order-invariant for multi-gene
    return "+".join(sorted(parts))


In [ ]:
deeper_res = res["deeper_analysis"]
metrics = ["pearson_delta", "pearson_delta_de"]

# map canonical pert id -> actual key used in deeper_res
canon2key = {}
for k in deeper_res.keys():
    canon2key.setdefault(canon_pert(k), k)

subgroup_analysis = {name: {m: [] for m in metrics}
                     for name in pert_data.subgroup["test_subgroup"].keys()}

missing = []

for name, pert_list in pert_data.subgroup["test_subgroup"].items():
    for pert in pert_list:
        ck = canon_pert(pert)
        key = canon2key.get(ck)
        if key is None:
            missing.append(pert)
            continue
        for m in metrics:
            subgroup_analysis[name][m].append(deeper_res[key][m])

if missing:
    print("Missing subgroup perts (first 20):", missing[:20], "count:", len(missing))

for name, result in subgroup_analysis.items():
    for m, vals in result.items():
        mean_value = np.mean(vals) if len(vals) > 0 else np.nan
        print(f"test_{name}_{m}: {mean_value}")


In [ ]:
# import pickle
# from pathlib import Path
# from datetime import datetime

# # current date + time
# timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# out_path = Path(f"./benchmarking_results/cellplm_perturbation_{data_name}_{timestamp}.pkl")

# with open(out_path, "wb") as f:
#     pickle.dump(res, f)

# print(f"Saved results to {out_path.resolve()}")


In [ ]:
pipe.fitted

In [ ]:
pipe.predict(
    pert_data=pert_data,
    # test_loader=test_loader_gears,
    device=DEVICE,
)

In [ ]:
save_dir ='./benchmarking_results'

In [ ]:
def plot_perturbation_cellplm_scgpt_style(
    pipe,
    pert_data,
    query: str,
    save_file: str = None,
    pool_size: int = None,
    loader=None,
    de_key: str = "top_non_dropout_de_20",
):
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns

    # --- match scGPT styling ---
    sns.set_theme(style="ticks", rc={"axes.facecolor": (0, 0, 0, 0)}, font_scale=1.5)

    adata = pert_data.adata

    # condition -> condition_name mapping (scGPT uses .values without drop_duplicates;
    # we'll keep it safe)
    cond2name = dict(adata.obs[["condition", "condition_name"]].drop_duplicates().values)
    if query not in cond2name:
        raise KeyError(f"Query '{query}' not found in adata.obs['condition'].")

    cond_name = cond2name[query]

    # --- DE list ---
    if de_key not in adata.uns or cond_name not in adata.uns[de_key]:
        raise KeyError(f"Missing DE list: adata.uns['{de_key}']['{cond_name}']")

    de_list = list(adata.uns[de_key][cond_name])  # typically ENSG IDs

    # --- map DE genes to var indices ---
    var_names = np.array(adata.var_names).astype(str)
    name_to_idx = {g: i for i, g in enumerate(var_names)}
    de_idx = np.array([name_to_idx[str(g)] for g in de_list], dtype=int)

    # --- display gene symbols like scGPT (gene_raw2id) ---
    if "gene_name" in adata.var.columns:
        gene_symbols = np.array(adata.var["gene_name"].astype(str).values)
        genes = [gene_symbols[i] for i in de_idx]
    else:
        genes = [str(g) for g in de_list]  # fallback

    # --- ground truth (gene space) ---
    truth = adata[adata.obs["condition"] == query].X
    truth = truth.toarray() if hasattr(truth, "toarray") else np.asarray(truth)
    truth = truth[:, de_idx]  # [n_cells, n_de]

    # --- control mean (scGPT uses to_df().mean(); we do equivalent) ---
    ctrl = adata[adata.obs["condition"] == "ctrl"].X
    ctrl = ctrl.toarray() if hasattr(ctrl, "toarray") else np.asarray(ctrl)
    ctrl_means = ctrl.mean(axis=0)[de_idx]  # [n_de]

    # --- CellPLM prediction ---
    # IMPORTANT: pipe.predict expects pert_data.dataloader unless you pass loader.
    if loader is None:
        assert hasattr(pert_data, "dataloader") and pert_data.dataloader is not None, \
            "pert_data.dataloader not found. Call pert_data.get_dataloader(..., no_splits=True) first."
        loader = pert_data.dataloader["test_loader"]

    out = pipe.predict(pert_data=pert_data, loader=loader)
    pred_mat = np.asarray(out["pred"])               # [B, G_model]
    pert_cat = np.asarray(out["pert_cat"]).astype(str)
    model_genes = np.asarray(out["model_genes"]).astype(str)

    # pool prediction for this query in model-gene space
    m = (pert_cat == query)
    if not np.any(m):
        raise ValueError(f"No predictions found for query '{query}' in the provided loader.")
    pred_q_model = pred_mat[m].mean(axis=0)          # [G_model]

    # map model genes -> adata genes
    gears_index = {g: i for i, g in enumerate(var_names)}
    pred_gene = np.full((adata.n_vars,), np.nan, dtype=np.float32)
    for j, g in enumerate(model_genes):
        i = gears_index.get(g)
        if i is not None:
            pred_gene[i] = pred_q_model[j]

    pred = pred_gene[de_idx]  # [n_de]

    # --- scGPT-style centering by control mean ---
    pred = pred - ctrl_means
    truth = truth - ctrl_means

    # --- plot using the SAME primitives as scGPT ---
    fig, ax = plt.subplots(figsize=[16.5, 4.5])
    plt.title(query)
    plt.boxplot(truth, showfliers=False, medianprops=dict(linewidth=0))

    for i in range(pred.shape[0]):
        _ = plt.scatter(i + 1, pred[i], color="red")

    plt.axhline(0, linestyle="dashed", color="green")
    ax.xaxis.set_ticklabels(genes, rotation=90)

    plt.ylabel("Change in Gene Expression over Control", labelpad=10)
    plt.tick_params(axis="x", which="major", pad=5)
    plt.tick_params(axis="y", which="major", pad=5)
    sns.despine()

    if save_file:
        fig.savefig(save_file, bbox_inches="tight", transparent=False, dpi=300)

    return fig


In [ ]:
pert_data.get_dataloader(batch_size=64, test_batch_size=64)
plot_loader = pert_data.dataloader["train_loader"]

plot_perturbation_cellplm_scgpt_style(
    pipe=pipe,
    pert_data=pert_data,
    query="DAD1+ctrl",
    loader=plot_loader,
    save_file=f"{save_dir}/{data_name}_DAD1+ctrl.png",
)

## Save checkpoint

In [ ]:
from CellPLM.pipeline.perturbation_prediction import PerturbationPredictionDefaultModelConfig
import json

model_config = PerturbationPredictionDefaultModelConfig.copy()

In [ ]:
checkpoint = {
    'model_state_dict': pipe.model.state_dict(),  # the model's weights
    'config': model_config  # model configuration (can be any object with config details)
}

checkpoint_path = f'./ckpt/perturbation_{data_name}.best.ckpt'
config_path = f'./ckpt/perturbation_{data_name}.config.json'

# Save the checkpoint (model weights, optimizer state)
torch.save(checkpoint, checkpoint_path)

# # Save the model config (e.g., JSON file)
with open(config_path, 'w') as f:
    json.dump(model_config, f) 

## Load checkpoint

In [ ]:
from CellPLM.pipeline.perturbation_prediction import PerturbationPredictionPipeline
from CellPLM.utils import set_seed

FINETUNE_VERSION = '20250620_ATAA'
DEVICE = 'cuda:0'

set_seed(42)

# data = ad.read_h5ad('../data/gse155468.h5ad')

pipeline = PerturbationPredictionPipeline(pretrain_prefix=FINETUNE_VERSION, # Specify the pretrain checkpoint to load
                                 pretrain_directory='../ckpt')
pipeline.model

In [ ]:
pipeline.fitted

In [ ]:
# pipeline.predict(
#     pert_data=pert_data,
#     # test_loader=test_loader_gears,
#     device=DEVICE,
# )